In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, KFold, cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_squared_error, mean_absolute_error

from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
from scipy.stats import randint, uniform

import os
import joblib

In [12]:
# 1- xgboost
# 2- adaboost
# 3- gradient boosting
# 4- catboost
# 5- lightboost
# 6- random forest
# 7- stacking 01
# 8- stacking 02

In [13]:
df = pd.read_csv("../data/preprocessed_dataset.csv")

df.head(2)

,FA (kg/m3),GGBFS (kg/m3),Coarse aggregate (kg/m3),Fine aggregate (kg/m3),Na2SiO3,NaOH,Sio2 (Dry),Concentration (M) NaOH,Superplasticizer,Total water,Initial curing time (day),Initial curing temp (C),Initial curing rest time (day),Final curing temp (C),Water_to_Binder,Water_to_Activator,SP_ratio,total_material_mass,Total_CO2_kg,Compressive_strength_MPa
0,0.000000,0.069565,-0.127973,-0.495765,0.700467,0.741549,0.756779,0.0,0.095238,0.651360,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.107509,0.885142,512.210033,46.57
1,0.140762,0.162319,-0.465568,0.198785,0.900600,0.950927,0.972882,0.0,0.142857,0.849623,0.0,0.0,0.0,0.0,0.629716,-0.436364,0.159822,0.948367,534.887466,46.01


In [14]:
labels = df[['Total_CO2_kg', 'Compressive_strength_MPa']]
df.drop(columns=['Total_CO2_kg', 'Compressive_strength_MPa'], inplace=True)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(df, labels, random_state=42, test_size=0.20)

In [16]:
X_train = X_train.copy()
X_test  = X_test.copy()
X_train.columns = X_train.columns.str.replace(r"\s+", "_", regex=True)
X_test.columns  = X_test.columns.str.replace(r"\s+", "_", regex=True)


# lightBoost

In [17]:
lgbm = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=1 
)

model = MultiOutputRegressor(lgbm)

param_dist = {
    "estimator__n_estimators": randint(200, 1500),
    "estimator__learning_rate": uniform(0.01, 0.29), 

    "estimator__num_leaves": randint(16, 128),
    "estimator__max_depth": [-1, 3, 5, 7],
    "estimator__min_child_samples": randint(2, 30),

#     "estimator__subsample": uniform(0.6, 0.4),
#     "estimator__colsample_bytree": uniform(0.6, 0.4),

    "estimator__subsample_freq": randint(0, 8),

    "estimator__reg_alpha": uniform(0.0, 2.0),
    "estimator__reg_lambda": uniform(0.0, 5.0),
}

cv = KFold(n_splits=10, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=100,
    scoring="r2",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=4,
    error_score="raise"
)

random_search.fit(X_train, y_train)

print("Best Hyperparameters:", random_search.best_params_)
print("Best R2:", random_search.best_score_)


Fitting 10 folds for each of 100 candidates, totalling 1000 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000068 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 469
[LightGBM] [Info] Number of data points in the train set: 220, number of used features: 18
[LightGBM] [Info] Start training from score 433.158628
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Ligh

In [19]:
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV, KFold

best = random_search.best_params_

base_lgbm = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=1
)

model = MultiOutputRegressor(base_lgbm)

grid = {
    "estimator__n_estimators": sorted(set([
        max(100, best["estimator__n_estimators"] - 200),
        best["estimator__n_estimators"],
        best["estimator__n_estimators"] + 200,
    ])),

    "estimator__learning_rate": sorted(set([
        max(0.005, best["estimator__learning_rate"] * 0.85),
        best["estimator__learning_rate"],
        min(0.5, best["estimator__learning_rate"] * 1.15),
    ])),

    "estimator__num_leaves": sorted(set([
        max(8, best["estimator__num_leaves"] - 32),
        best["estimator__num_leaves"],
        best["estimator__num_leaves"] + 32,
    ])),

    "estimator__max_depth": sorted(set([
        max(-1, best["estimator__max_depth"] - 2),
        best["estimator__max_depth"],
        best["estimator__max_depth"] + 2,
    ])),

    "estimator__min_child_samples": sorted(set([
        max(2, best["estimator__min_child_samples"] - 10),
        best["estimator__min_child_samples"],
    ])),

    "estimator__subsample": [best.get("estimator__subsample", 0.8)],
    "estimator__colsample_bytree": [best.get("estimator__colsample_bytree", 0.8)],
    "estimator__subsample_freq": [best.get("estimator__subsample_freq", 0)],
    "estimator__reg_alpha": [best.get("estimator__reg_alpha", 0.0)],
    "estimator__reg_lambda": [best.get("estimator__reg_lambda", 0.0)],
}

cv = KFold(n_splits=10, shuffle=True, random_state=42)

n_candidates = 1
for v in grid.values():
    n_candidates *= len(v)
print("Grid candidates:", n_candidates, "| Total fits:", n_candidates * cv.get_n_splits())

grid_search = GridSearchCV(
    estimator=model,
    param_grid=grid,
    scoring="r2",
    cv=cv,
    verbose=2,
    n_jobs=4,
    error_score="raise"
)

grid_search.fit(X_train, y_train)

print("Best params (Grid):", grid_search.best_params_)
print("Best R2 (Grid):", grid_search.best_score_)

Grid candidates: 162 | Total fits: 1620
Fitting 10 folds for each of 162 candidates, totalling 1620 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000058 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 469
[LightGBM] [Info] Number of data points in the train set: 220, number of used features: 18
[LightGBM] [Info] Start training from score 433.158628
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

In [20]:
best_model = grid_search.best_estimator_

cv = KFold(n_splits=10, shuffle=True, random_state=42)

scoring = {
    "r2": "r2",
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error"
}

cv_res = cross_validate(
    best_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=4,
    return_train_score=False
)

print("CV R2 mean ± std:", cv_res["test_r2"].mean(), "±", cv_res["test_r2"].std())
print("CV RMSE mean ± std:", (-cv_res["test_rmse"]).mean(), "±", (-cv_res["test_rmse"]).std())
print("CV MAE mean ± std:", (-cv_res["test_mae"]).mean(), "±", (-cv_res["test_mae"]).std())


CV R2 mean ± std: 0.9625229654659426 ± 0.013390227053101014
CV RMSE mean ± std: 3.9926875893639187 ± 1.148162523012893
CV MAE mean ± std: 2.503121509639768 ± 0.48539079116210443


In [21]:
model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000113 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 469
[LightGBM] [Info] Number of data points in the train set: 220, number of used features: 18
[LightGBM] [Info] Start training from score 433.158628
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

,estimator,LGBMRegressor...ndom_state=42)
,n_jobs,None
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0


In [22]:
y_pred = model.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
mae_test = mean_absolute_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)

print("TEST R2:", r2_test)
print("TEST RMSE:", rmse_test)
print("TEST MAE:", mae_test)

TEST R2: 0.9520323537956685
TEST RMSE: 4.428561027491846
TEST MAE: 3.2198324583042295


In [23]:
model_dir = "../ml_models"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, "lightBoost_final_model.pkl")
joblib.dump(best_model, model_path)  

print(f"Model saved to: {model_path}")


Model saved to: ../ml_models\lightBoost_final_model.pkl
